In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import duckdb
import numpy as np
from mplsoccer import Pitch, VerticalPitch
from plottable import ColumnDefinition, Table
from plottable.cmap import normed_cmap
from matplotlib.colors import PowerNorm
from matplotlib.lines import Line2D
from matplotlib.lines import Line2D
from mplsoccer import VerticalPitch
import matplotlib.patches as mpatches
import math

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

from great_tables import GT

from extract_carries import extract_carries
import xg_model_baked

*Create metrics class*
        
        *Subtask: simple xG model*
        *Subtask: extrapolate carries*

Create Z-Score Creation Class

Create Export Table Class

Forwards
    npXg
    xG assisted (cba figuring out xA fully, will never have to do that in the real world)
    *shots*
    *npGoals*
    *Aerial %*
    *EPV passing*
    *Dribble/Takeon rate*
    *Carrying*
    *Touches in final third*
    Through balls received
    *Through balls played*
    *Fouls won*

Midfielders
    *Progressive pass distance*
    *Progressive passes*
    *Progressive carries*
    *Passes into penalty area*
    *Total tackles*
    *Successful tackles*
    *Tackle %*
    *Pass completion %*
    *Forward passes*
    *Forward pass %*
    *Interceptions (tackles and interceptions?)*
    *Touches*


Defenders
    *Blocked shots*
    *Recoveries*
    *Blocked Crosses*
    *Clearances*

In [3]:
class Metrics:
    def __init__(self, file_path: str, conditions: str = None):
        """
        file_path is path to DuckDB database. conditions is a string for the
        WHERE clause if a specific range of events is needed.
        """
        con = duckdb.connect(file_path)
        if conditions is None:
            self.events = con.sql("select * from events").df()
        else:
            self.events = con.sql(f"select * from events where {conditions}").df()
        self.events['playerId'] = self.events['playerId'].fillna(-1).astype('int64')  # keep start/end slides for through-ball math, give them an int id
        self.players = con.sql("select * from players").df()
        con.close()

        self.events = xg_model_baked.add_xg(self.events)

    def create_metrics(self):
        ev = self.events
        is_shot = ev["type"].isin(["Goal", "SavedShot", "MissedShots", "ShotOnPost"])
        pens = ev[["penaltyScored", "penaltyMissed"]].fillna(False).astype(bool).any(axis=1)

        shots = ev[is_shot]
        shooting_agg = (
            shots.assign(np_shot=(~pens[is_shot]).astype(int),
                         np_goal=((shots["type"] == "Goal") & ~pens[is_shot]).astype(int))
            .groupby("playerId")
            .agg(
                shots=("np_shot", "sum"),
                np_goals=("np_goal", "sum"),
                npxg=("xG", "sum"),
                big_chances_missed=("bigChanceMissed", "sum"),
                big_chances_scored=("bigChanceScored", "sum"),
            )
        )
        shooting_agg["big_chances"] = (
            shooting_agg["big_chances_missed"] + shooting_agg["big_chances_scored"]
        )

        dribbling_agg = (
            ev.groupby("playerId")
            .agg(
                dribbles_lost=("dribbleLost", "sum"),
                dribbles_won=("dribbleWon", "sum"),
            )
        )
        dribbling_agg["dribbles_total"] = (
            dribbling_agg["dribbles_lost"] + dribbling_agg["dribbles_won"]
        )
        dribbling_agg["dribble_success"] = (
            dribbling_agg["dribbles_won"] / dribbling_agg["dribbles_total"].replace(0, np.nan)
        ) * 100

        passing_df = ev[ev["type"] == "Pass"].copy()
        keypass_cols = ["keyPassLong", "keyPassShort", "keyPassCross",
                        "keyPassThroughball", "keyPassOther"]
        passing_df["isKeyPassOP"] = passing_df[keypass_cols].fillna(False).any(axis=1)
        passing_df["isCompleted"] = passing_df["outcomeType"] == "Successful"

        passing_agg_basic = (
            passing_df.groupby("playerId")
            .agg(
                key_passes_open_play=("isKeyPassOP", "sum"),
                through_balls_completed=("passThroughBallAccurate", "sum"),
                passes=("isTouch", "sum"),
                completed_passes=("isCompleted", "sum"),
                pass_epv=("EPV", "sum"),
                forward_passes=("passForward", "sum"),
                big_chances_created=("bigChanceCreated", "sum"),
                xg_assisted=("xG_assisted", "sum"),
            )
        )
        passing_agg_basic["completion_percentage"] = (
            passing_agg_basic["completed_passes"]
            / passing_agg_basic["passes"].replace(0, np.nan)
        ) * 100
        passing_agg_basic["forward_pct"] = (
            passing_agg_basic["forward_passes"]
            / passing_agg_basic["passes"].replace(0, np.nan)
        ) * 100

        adv = ev[(ev["type"] == "Pass") & (ev["outcomeType"] == "Successful")].copy()
        adv["dist_start"] = np.sqrt((100 - adv["x"]) ** 2 + (50 - adv["y"]) ** 2)
        adv["dist_end"] = np.sqrt((100 - adv["endX"]) ** 2 + (50 - adv["endY"]) ** 2)
        adv["progressive_passing_distance"] = (adv["endX"] - adv["x"]).clip(lower=0.0)
        adv["isProgressivePass"] = (adv["x"] > 33.333333) & (
            adv["dist_start"] <= 0.75 * adv["dist_end"]
        )
        adv["penAreaEnd"] = (adv["endX"] >= 83) & (adv["endY"] >= 21) & (adv["endY"] <= 79)

        passing_agg_advanced = (
            adv.groupby("playerId")
            .agg(
                progressive_passing_distance=("progressive_passing_distance", "sum"),
                progressive_passes=("isProgressivePass", "sum"),
                passes_into_pen_area=("penAreaEnd", "sum"),
            )
        )

        defense_agg = (
            ev.groupby("playerId")
            .agg(
                interceptions=("interceptionWon", "sum"),
                tackles_won=("tackleWon", "sum"),
                tackles_lost=("tackleLost", "sum"),
                errors_leading_to_goal=("errorLeadsToGoal", "sum"),
                errors_leading_to_shot=("errorLeadsToShot", "sum"),
                clearances=("clearanceEffective", "sum"),
                recoveries=("ballRecovery", "sum"),
                blocks=("outfielderBlock", "sum"),
                blocked_crosses=("passCrossBlockedDefensive", "sum"),
                aerial_wins=("duelAerialWon", "sum"),
                aerial_losses=("duelAerialLost", "sum"),
                interceptions_in_box=("interceptionIntheBox", "sum"),
            )
        )
        defense_agg["tackles_total"] = defense_agg["tackles_won"] + defense_agg["tackles_lost"]
        defense_agg["tackle_success"] = (
            defense_agg["tackles_won"] / defense_agg["tackles_total"].replace(0, np.nan)
        ) * 100
        defense_agg["aerials"] = defense_agg["aerial_wins"] + defense_agg["aerial_losses"]
        defense_agg["aerial_success"] = (
            defense_agg["aerial_wins"] / defense_agg["aerials"].replace(0, np.nan)
        ) * 100

        touches = ev[ev["isTouch"] == True]
        touches_agg = (
            touches.groupby("playerId")
            .agg(
                touches=("playerId", "count"),
                touches_in_final_third=("finalThird", "sum"),
                fouls_won=("foulGiven", "sum"),
            )
        )

        carries = extract_carries(ev, 2.0)
        carries["progressive_carrying_distance"] = (
            carries["endX"] - carries["startX"]
        ).clip(lower=0.0)
        carries["isProgressive"] = (
            ((carries["startX"] < 60) & (carries["endX"] >= 10.5 + carries["startX"]))
            | ((carries["startX"] > 60) & (carries["endX"] >= 5.25 + carries["startX"]))
            | ((carries["endX"] >= 83) & (carries["endY"] >= 21) & (carries["endY"] <= 79))
        )
        carries["intoPenArea"] = (
            (carries["endX"] >= 83) & (carries["endY"] >= 21) & (carries["endY"] <= 79)
        )
        carries_agg = (
            carries.groupby("playerId")
            .agg(
                progressive_carrying_distance=("progressive_carrying_distance", "sum"),
                progressive_carries=("isProgressive", "sum"),
                carries_into_penalty_area=("intoPenArea", "sum"),
            )
        )

        player_info = self.players[['playerId', 'playerName', 'teamName', 'minutes', 'position']]
        player_info = player_info.set_index('playerId')

        metrics = pd.concat(
            [player_info, shooting_agg, dribbling_agg, passing_agg_basic,
             passing_agg_advanced, defense_agg, touches_agg, carries_agg],
            axis=1,
        )

        metrics = metrics.drop(index=[-1, 0], errors='ignore')

        return metrics
    
    def generate_normalized_metrics(self):
        """
        normalizes applicable metrics to per 90 (figuring out TIP/Otip soon)
        """
        df = self.create_metrics().copy()

        
        df['minutes'] = df['minutes'].astype(str).str.replace(',', '').astype(float)

        exclude = {
        'playerName', 'teamName', 'minutes',
        'completion_percentage', 'forward_pct', 'dribble_success',
        'tackle_success', 'aerial_success',
            }
        cols = [c for c in df.columns
            if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]

        nineties = df['minutes'] / 90
        df[cols] = df[cols].div(nineties, axis=0)
        
        return df


        





In [ ]:
obj = Metrics('/Users/owner/Desktop/Natabase/championship_2526.duckdb')

df1 = obj.generate_normalized_metrics()


/Users/owner/Desktop/GitHub/NextStepsFootballAnallytics/CompRatingArticle/MLSCompRatingArticle/xg_model_baked.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prev = ev.groupby("matchId")["keyPassThroughball"].shift(1).fillna(False).astype(int)


KeyboardInterrupt: 